# Transform ecg photo to numerical

## Requirement

In [ ]:
pip install 

In [ ]:
pip install pandas

In [ ]:
pip install numpy

In [1]:
!pip install opencv-python


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
!pip install albumentations

  Obtaining dependency information for albumentations from https://files.pythonhosted.org/packages/8e/64/013409c451a44b61310fb757af4527f3de57fc98a00f40448de28b864290/albumentations-2.0.8-py3-none-any.whl.metadata
  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Obtaining dependency information for scipy>=1.10.0 from https://files.pythonhosted.org/packages/df/75/b4ce781849931fef6fd529afa6b63711d5a733065722d0c3e2724af9e40a/scipy-1.17.1-cp311-cp311-macosx_10_14_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.8 MB/s eta 0:00:00
  Obtaining dependency information for pydantic>=2.9.2 from https://files.pythonhosted.org/packages/5a/87/b70ad306ebb6f9b585f114d0ac2137d792b48be34d732d60e597c2f8465a/pydantic-2.12.5-py3-none-any.whl.metadata
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Obtaining dependency information for albucore==0.0.24 from https://files.pythonhosted.org/packages/0a/e2/91f145e1f32428e9e1f21f46a7022ffe6

In [1]:
from transformers import pipeline
print("OK")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


OK


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import albumentations as A

import tqdm
from scipy.signal import find_peaks

ModuleNotFoundError: No module named 'albumentations'

In [ ]:
import cv2

## Data of each lead with PR_ms,QRS_ms, QT_ms, QTc_ms, n_beats from ecg photos

In [ ]:
# List of names of sub-repertory in inference_inputs
name_repertory1=[]
repertory1=r"C:\Users\Bureau\Desktop\ecg_pipeline\data\inference_inputs"
for file in os.listdir(repertory1):
    #path_repertory1.append(os.path.join(repertory1, file))
    name_repertory1.append(file)
#path_repertory[0:5]
name_repertory1[0:5]

In [ ]:
print(f"The number of sub-repertory name is : {len(name_repertory1)}" 

In [ ]:
# List of names of files in each sub-repertory
complete_mane=[]
for i in name_repertory1:
    repertory2 = rf"C:\Users\Bureau\Desktop\ecg_pipeline\data\inference_inputs\{i}"
    for file in os.listdir(repertory2):
        
        complete_mane.append(file)
complete_mane[-10 :]

In [ ]:
print(f"The number of photo ecg name with csv file is : {len(complete_mane)}" 

In [ ]:
name_without_csv = [x for x in complete_mane if not x.endswith("csv")]
name_without_csv[-10:]

In [ ]:
print(f"The number of photo ecg name without csv file is : {len(name_without_csv)}" 

In [ ]:
# Translate each photo to numerical with all leads and save in the repertory to outputs   
for name_file in name_repertory1:
    for name_photo in name_without_csv:
        if name_file == name_photo.split('-')[0]:
            !python infer_ecg.py --img data/inference_inputs/{name_file}/{name_photo} --out outputs


## New data with averages of PR_ms,QRS_ms, QT_ms, QTc_ms, n_beats

In [ ]:
#List names of files in repertory who contain csv files
name_repertory_csv=[]
repertory_output=r"C:\Users\Bureau\Desktop\ecg_pipeline\outputs"
for file in os.listdir(repertory_output):
    name_repertory_csv.append(file)
name_repertory_csv[0:5]

In [ ]:
# List with only csv files
name_repertory_csv1 = [x for x in name_repertory_csv if x.endswith("csv")]
name_repertory_csv1[0:5]

In [ ]:
# To make a data with average of columns of numerical csv files 
path=name_repertory_csv1[0]
df=pd.read_csv(f"outputs\{path}")
df=df.drop("lead", axis=1)
df = df.apply(lambda col: col.mean()).to_frame().T
for path in name_repertory_csv1[1:]:
    dfi=pd.read_csv(f"outputs\{path}")
    dfi=dfi.drop("lead", axis=1)
    dfi = dfi.apply(lambda col: col.mean()).to_frame().T
    df = pd.concat([df, dfi], axis=0, ignore_index=True)

# To save df as csv file
df.to_csv("ecg_dg.csv", index=False)

In [ ]:
# It's possible to write this function in the file untils.py and import from there for using like pandas

In [ ]:
# Function to calculate the heart rate of an ecg photo
def heart(ecg_name):
    #define the image
    img= cv2.imread(ecg_name)
    #translate to gray
    gray=cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5),0)
    #Detection of signal (projection) 
    signal =np.mean(255 - blur, axis=0)
    #Detection of peaks
    peaks, _ =find_peaks(signal, distance=50, height=np.percentile(signal, 90))
    # Calcul RR
    rr_intervals = np.diff(peaks) #in pixel
    #translate pixel to time(100 pixels = 1 sec)
    seconds_per_pixel =1/100
    rr_seconds = rr_intervals*seconds_per_pixel
    # HR
    hr=60 / np.mean(rr_seconds)
    return hr

In [ ]:
#To make the list heart rate HR of ecg photos 
HR=[]
for name_file in name_repertory1:
    for name_photo in name_without_csv:
        if name_file == name_photo.split('-')[0]:
            ecg_name=rf"data/inference_inputs/{name_file}/{name_photo}"
            HR.append(heart(ecg_name))
HR[:10]

In [ ]:
# To translate list HD to dataframe
HR = pd.DataFrame(HR)

In [ ]:
# It's possible to write this function in the file untils.py and import from there for using like pandas

In [ ]:
# Function to calculate the proportion artifact of an ecg photo
def artefact1(ecg_name):
    #define the image
    img= cv2.imread(ecg_name)
    #translate to gray
    gray=cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5),0)
    #Detection of signal (projection) 
    signal =np.mean(255 - blur, axis=0)
    #Detection of peaks
    peaks, _ =find_peaks(signal, distance=50, height=np.percentile(signal, 90))
    # Calcul RR
    rr_intervals = np.diff(peaks) #in pixel
    #translate pixel to time(100 pixels = 1 sec)
    seconds_per_pixel =1/100
    rr_seconds = rr_intervals*seconds_per_pixel
    # Artefacts
    artefacts_rr = rr_seconds[(rr_seconds < 0.3) | (rr_seconds > 2.5)]    
    return len(artefacts_rr)/len(rr_seconds) if len(rr_seconds)!=0 else "Nan"

In [ ]:
# To make the list of artifacts signals  
artifact=[]
for name_file in name_repertory1:
    for name_photo in name_without_csv:
        if name_file == name_photo.split('-')[0]:
            ecg_name=rf"data/inference_inputs/{name_file}/{name_photo}"
            artifact.append(artefact1(ecg_name))
artifact[:10]

In [ ]:
# To translate list artifact to dataframe
artifact = pd.DataFrame(artifact)

In [ ]:
# To add the columns HR(heart rate) and artifact%(artifact signal) at df 
df["HR"]=HR[0]
df["artifact%"]=artifact[0]

In [ ]:
df

In [ ]:
# to save the final numerical data
df.to_csv("ecg_dg_final.csv", index=False)

In [ ]:
# to show how many miss value
df.isna().sum()

In [ ]:
# The data df without the zero lines of columns artifact%
df[df["artifact%"] !=0]